# 06 - Spectrogram Comparison: clean vs noisy vs combined_wiener vs learned

Figure-source notebook (03/04 style). On the FROZEN synthetic test pairs
(`data/synth_pairs/test`) we have the **clean ground truth**, so each example
shows 4 panels — clean | noisy | combined_wiener | learned Wave-U-Net — and you
can SEE which method's output lands closest to true clean.

Ties the metrics together visually: SI-SDR (learned 5.27 > cw 1.44) +
DNSMOS-BAK (learned 3.50 > cw 3.35) said the learned model both preserves signal
AND removes noise better; the user's listen agreed (learned a little higher MOS).
Expectation in the spectrograms: **combined_wiener removes aggressively** (visible
band-limiting / holes), **learned is gentler / closer to clean**.

Per-panel: log-power spectrogram (0-4 kHz, the clinical band) + an audio player.
Δ-vs-clean (SI-SDR) annotated per method so the figure is self-documenting.

In [ ]:
import sys
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from hydra import compose, initialize
from hydra.utils import instantiate
from omegaconf import OmegaConf
from IPython.display import Audio, HTML, display

REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
PAIRS = REPO_ROOT / 'data' / 'synth_pairs' / 'test'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

manifest = pd.read_csv(PAIRS / 'manifest.csv')
manifest['fname'] = (manifest['segment_id'].str.replace('/', '_')
                     .str.replace(':', '_') + '.wav')
print(len(manifest), 'test pairs')

In [ ]:
# --- load the two methods: learned Wave-U-Net + combined_wiener (44.1k) ---
ckpt = torch.load('../saved/wun_sisdr/model_best.pth', map_location=DEVICE)
ccfg = ckpt['config'] if OmegaConf.is_config(ckpt['config']) else OmegaConf.create(ckpt['config'])
learned = instantiate(ccfg.model).to(DEVICE)
learned.load_state_dict(ckpt['state_dict'])
learned.eval()

with initialize(version_base=None, config_path='../src/configs'):
    cw_cfg = compose(config_name='inference',
                     overrides=['model=combined_wiener', 'datasets.test.target_sr=44100'])
combined_wiener = instantiate(cw_cfg.model).to(DEVICE)
combined_wiener.eval()
print('learned:', learned)
print('combined_wiener loaded')

In [ ]:
def _si_sdr(est, ref, eps=1e-8):
    est = est - est.mean(); ref = ref - ref.mean()
    a = (est * ref).sum() / (np.sum(ref ** 2) + eps)
    proj = a * ref
    return 10 * np.log10((np.sum(proj ** 2) + eps) / (np.sum((est - proj) ** 2) + eps))


def run_learned(noisy, sr):
    with torch.no_grad():
        x = torch.from_numpy(noisy.astype('float32'))[None, None].to(DEVICE)
        return learned(audio=x)['audio'][0, 0].cpu().numpy()


def run_cw(noisy, sr):
    # combined_wiener runs at 44.1k; resample around it
    n44 = librosa.resample(noisy, orig_sr=sr, target_sr=44100)
    with torch.no_grad():
        x = torch.from_numpy(n44.astype('float32'))[None, None].to(DEVICE)
        o = combined_wiener(audio=x)['audio'][0, 0].cpu().numpy()
    return librosa.resample(o, orig_sr=44100, target_sr=sr)[:len(noisy)]


def compare(row, fmax=4000):
    clean, sr = sf.read(PAIRS / 'clean' / row['fname'], dtype='float32')
    noisy, _ = sf.read(PAIRS / 'noisy' / row['fname'], dtype='float32')
    learned_o = run_learned(noisy, sr)
    cw_o = run_cw(noisy, sr)

    n = min(map(len, [clean, noisy, learned_o, cw_o]))
    clean, noisy, learned_o, cw_o = (a[:n] for a in (clean, noisy, learned_o, cw_o))
    panels = {
        'clean (GT)': clean,
        f'noisy ({_si_sdr(noisy, clean):.1f} dB)': noisy,
        f'combined_wiener ({_si_sdr(cw_o, clean):.1f} dB)': cw_o,
        f'learned ({_si_sdr(learned_o, clean):.1f} dB)': learned_o,
    }
    display(HTML(f"<b>{row['segment_id']} | {row['donor_category']} | "
                 f"SNR {row['snr_db']:.0f} dB | sr={sr}</b> "
                 f"<span style='color:gray'>(panel titles = SI-SDR vs clean; higher=closer)</span>"))
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
    for ax, (name, y) in zip(axes, panels.items()):
        ax.specgram(y, Fs=sr, NFFT=512, noverlap=384, cmap='magma')
        ax.set_title(name, fontsize=9)
        ax.set_ylim(0, min(fmax, sr // 2))
        ax.label_outer()
    plt.tight_layout(); plt.show()
    for name, y in panels.items():
        display(HTML(f'<i>{name}</i>')); display(Audio(y, rate=sr))

## Examples across noise categories and SNRs

A spread: lowest-SNR (hardest) per donor category, where the methods differ most.

In [ ]:
sample = manifest.sort_values('snr_db').groupby('donor_category').head(1)
for _, row in sample.iterrows():
    compare(row)

In [ ]:
# a few easier (higher-SNR) examples too, for contrast
for _, row in manifest.sort_values('snr_db', ascending=False).head(3).iterrows():
    compare(row)

## Notes / takeaways

_Fill in after viewing:_
- Does combined_wiener show visible band-limiting / holes (aggressive removal)?
- Does the learned output look closest to the clean GT panel?
- Where the SI-SDR numbers in the titles confirm/contradict the visual.
- Pick 2-3 clean example figures for the thesis (one per noise type).